In [1]:
%matplotlib inline

import sys
import importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import detector_model_v4 as dm
import time
import csv

def measure(function, repeats=3):
    function()  # прогрев
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        function()
        times.append(time.perf_counter() - start)
    return float(np.median(times))

detector = dm.Detector()
M = 3000
seed = 123
directions = dm.random_directions(M, np.random.default_rng(seed))
position = np.array([0.2, -0.3, 0.7])
timings = []

def record(stage, slow, fast):
    t_loop = measure(slow)
    t_array = measure(fast)
    timings.append([stage, t_loop, t_array, t_loop / t_array])
    print(f"{stage}: цикл={t_loop:.6f} с; NumPy={t_array:.6f} с; "
          f"ускорение={t_loop/t_array:.2f}")

In [2]:
def generate_loop():
    rng = np.random.default_rng(seed)
    return np.array([dm.random_direction(rng) for _ in range(M)])

def generate_array():
    return dm.random_directions(M, np.random.default_rng(seed))

assert np.allclose(generate_loop(), generate_array())
record("generation", generate_loop, generate_array)

generation: цикл=0.011877 с; NumPy=0.000144 с; ускорение=82.25


**Сравним распространение и поиск попаданий**

In [3]:
points, distance = dm.sphere_intersections(position, directions, detector.R)
p_loop, s_loop = dm.sphere_intersections_loop(position, directions, detector.R)
assert np.allclose(points, p_loop)
assert np.allclose(distance, s_loop)

record("propagation",
       lambda: dm.sphere_intersections_loop(position, directions, detector.R),
       lambda: dm.sphere_intersections(position, directions, detector.R))

ids_loop = detector.find_pmt_loop(points)
ids_array = detector.find_pmt(points, method="full")
assert np.array_equal(ids_loop, ids_array)
record("pmt_full_search",
       lambda: detector.find_pmt_loop(points),
       lambda: detector.find_pmt(points, method="full"))

propagation: цикл=0.006917 с; NumPy=0.000245 с; ускорение=28.25
pmt_full_search: цикл=0.174509 с; NumPy=0.000286 с; ускорение=610.38


**Векторизуем подсчёт фотоэлектронов и весь проход**

In [4]:
rng = np.random.default_rng(987)
uniforms = rng.random(M)
pe_per_photon = rng.poisson(detector.a, size=M)

def count_loop(ids, survived, weights):
    signal = np.zeros(detector.N, dtype=np.int64)
    for j, alive, weight in zip(ids, survived, weights):
        if alive and j >= 0:
            signal[j] += weight
    return signal

def count_array(ids, survived, weights):
    selected = survived & (ids >= 0)
    return np.bincount(ids[selected], weights=weights[selected],
                       minlength=detector.N).astype(np.int64)

survived = uniforms < np.exp(-distance / 8.0)
assert np.array_equal(count_loop(ids_array, survived, pe_per_photon),
                      count_array(ids_array, survived, pe_per_photon))
record("photoelectron_count",
       lambda: count_loop(ids_array, survived, pe_per_photon),
       lambda: count_array(ids_array, survived, pe_per_photon))

def pipeline_loop():
    points, distances = dm.sphere_intersections_loop(position, directions, detector.R)
    ids = detector.find_pmt_loop(points)
    survived = uniforms < np.exp(-distances / 8.0)
    return count_loop(ids, survived, pe_per_photon)

def pipeline_array():
    points, distances = dm.sphere_intersections(position, directions, detector.R)
    ids = detector.find_pmt(points, method="full")
    survived = uniforms < np.exp(-distances / 8.0)
    return count_array(ids, survived, pe_per_photon)

assert np.array_equal(pipeline_loop(), pipeline_array())
record("propagate_search_count", pipeline_loop, pipeline_array)

photoelectron_count: цикл=0.000809 с; NumPy=0.000239 с; ускорение=3.38
propagate_search_count: цикл=0.191144 с; NumPy=0.000486 с; ускорение=393.14
